# Prompt engineering, one improvement at a time

We will ask the **same question about the same syllabus** repeatedly. Each step adds one useful part of a prompt and lets us compare the output.

## Setup

This notebook calls the Groq API. Add `GROQ_API_KEY` to the project `.env` file before running it.

In [1]:
%pip install -q openai python-dotenv

import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(Path.cwd().parent / '.env')  # running from notebooks/
load_dotenv(Path.cwd() / '.env')         # running from project root

client = OpenAI(
    api_key=os.environ['GROQ_API_KEY'],
    base_url='https://api.groq.com/openai/v1',
)
MODEL = os.getenv('GROQ_MODEL', 'llama-3.3-70b-versatile')

def ask(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.2,
    )
    return response.choices[0].message.content

print(f'Using Groq model: {MODEL}')

Note: you may need to restart the kernel to use updated packages.
Using Groq model: openai/gpt-oss-20b


## The supplied syllabus

This is the only source the model should use. The goal is to extract course outcomes that can actually be assessed.

In [2]:
SYLLABUS = '''
Course: Cloud-Native Application Development

Modules
1. REST API design, authentication, testing, and deployment.
2. Containers, orchestration, and continuous delivery.
3. Cloud-native systems and modern engineering practice.

Students will design and test a REST API with authentication.
Students will deploy a containerised service using a continuous delivery pipeline.
Students will gain exposure to cloud-native systems.
'''

print(SYLLABUS)


Course: Cloud-Native Application Development

Modules
1. REST API design, authentication, testing, and deployment.
2. Containers, orchestration, and continuous delivery.
3. Cloud-native systems and modern engineering practice.

Students will design and test a REST API with authentication.
Students will deploy a containerised service using a continuous delivery pipeline.
Students will gain exposure to cloud-native systems.



## 1. Start with a basic prompt

This is a reasonable request, but it leaves the model to decide what counts as an outcome and how much to invent.

In [ ]:
basic_prompt = f'''
Extract assessable course outcomes from this syllabus.

{SYLLABUS}
'''

print(ask(basic_prompt))

## 2. Add an evidence boundary

Now we tell the model what evidence it may use. It must not repair vague syllabus wording by inventing a stronger outcome.

In [ ]:
evidence_prompt = f'''
Extract assessable course outcomes from the syllabus below.
Use only statements explicitly present in the supplied syllabus.
Do not add, strengthen, or infer outcomes that are not stated.

<syllabus>
{SYLLABUS}
</syllabus>
'''

print(ask(evidence_prompt))

## 3. Add constraints

The model now has a clear decision rule: include only outcomes with an observable action; flag vague statements instead of pretending they are assessable.

In [ ]:
constrained_prompt = f'''
Extract assessable course outcomes from the syllabus below.

Rules:
- Use only statements explicitly present in the syllabus.
- An assessable outcome must contain an observable student action.
- Exclude vague statements such as 'gain exposure' from the outcome list.
- Put excluded statements in a separate 'Needs review' section.

<syllabus>
{SYLLABUS}
</syllabus>
'''

print(ask(constrained_prompt))

## 4. Add an output contract

A good answer is not enough when another person or program needs to use it. Here we specify exactly what the response should look like.

In [ ]:
contract_prompt = f'''
Extract assessable course outcomes from the syllabus below.

Rules:
- Use only statements explicitly present in the syllabus.
- An assessable outcome must contain an observable student action.
- Do not rewrite, strengthen, or infer outcomes.

Return exactly this format:
ASSESSABLE OUTCOMES
- <verbatim syllabus statement>

NEEDS REVIEW
- <verbatim syllabus statement> — <brief reason>

<syllabus>
{SYLLABUS}
</syllabus>
'''

print(ask(contract_prompt))

## 5. Add examples

Few-shot examples teach a boundary that is difficult to describe perfectly in rules. Notice that the examples are **not** copied from the syllabus being evaluated.

In [ ]:
few_shot_prompt = f'''
Extract assessable course outcomes from the syllabus below.
Use only statements explicitly present in the syllabus.

Examples:
Statement: Students will implement and test a database schema.
Decision: ASSESSABLE OUTCOME

Statement: Students will become familiar with data systems.
Decision: NEEDS REVIEW — no observable action is stated.

Return exactly this format:
ASSESSABLE OUTCOMES
- <verbatim syllabus statement>

NEEDS REVIEW
- <verbatim syllabus statement> — <brief reason>

<syllabus>
{SYLLABUS}
</syllabus>
'''

print(ask(few_shot_prompt))

## Try it yourself

Change exactly one thing at a time: add a new ambiguous syllabus statement, alter one constraint, or add one better example. Rerun the final cell and observe what changes.

The key lesson: prompt engineering is not a bag of magic phrases. It is progressively making the task, evidence, rules, output, and examples explicit.